# Wholesale Customer Segmentation
K-Means + Agglomerative Clustering on 6 spend columns.

## Step 1 — Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

df = pd.read_csv("dataset/raw_wholesale_customers.csv")
df.head()

In [ ]:
df.info()

## Step 2 — Select Features + IQR Cap

In [ ]:
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

X = df[FEATURES].copy()

def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in FEATURES:
    low, high = iqr_fun(X[col])
    X[col] = X[col].clip(lower=low, upper=high)

df[FEATURES] = X
X.describe().round(2)

## Step 3 — Scale Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled shape:", X_scaled.shape)

## Step 4 — Elbow Method

In [ ]:
sse = {}
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    sse[k] = km.inertia_

for k, val in sse.items():
    print(f"k={k}  SSE={val:.2f}")

## Step 5 — Train K-Means

In [ ]:
kmeans = KMeans(n_clusters=5, n_init="auto", random_state=42)
km_labels = kmeans.fit_predict(X_scaled)

df["KMeans_Cluster"] = km_labels.astype(int)
df["KMeans_Cluster"].value_counts().sort_index()

## Step 6 — Evaluate K-Means

In [ ]:
sil_km = silhouette_score(X_scaled, km_labels)
dbi_km = davies_bouldin_score(X_scaled, km_labels)

print(f"Silhouette Score : {sil_km:.3f}  (closer to +1 is better)")
print(f"Davies-Bouldin   : {dbi_km:.3f}  (lower is better)")

In [ ]:
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"
centers_df.round(2)

## Step 7 — Second Algorithm: Agglomerative Clustering

I chose **Agglomerative (Hierarchical) Clustering** as the second method. It builds clusters bottom-up by merging the closest pair of clusters at each step. Unlike K-Means it does not use centroids or require iterative updates — it just merges based on distance. This makes it a good comparison because it approaches the segmentation problem from a completely different angle.

For wholesale customers the spending patterns between groups are quite distinct (fresh food buyers vs grocery/retail buyers), which means a linkage-based method should find similar natural groupings to K-Means without needing the elbow method to choose k.

**Source:** scikit-learn documentation — [AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)

In [ ]:
agg = AgglomerativeClustering(n_clusters=5, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)

df["Agg_Cluster"] = agg_labels.astype(int)
df["Agg_Cluster"].value_counts().sort_index()

## Step 8 — Compare Methods

In [ ]:
sil_agg = silhouette_score(X_scaled, agg_labels)
dbi_agg = davies_bouldin_score(X_scaled, agg_labels)

print(f"K-Means       — Silhouette: {sil_km:.3f}  Davies-Bouldin: {dbi_km:.3f}")
print(f"Agglomerative — Silhouette: {sil_agg:.3f}  Davies-Bouldin: {dbi_agg:.3f}")

## Step 9 — Sanity Check (3 Clients)

In [ ]:
sample_idx = [0, 1, 2]
cols = ["Channel", "Region"] + FEATURES + ["KMeans_Cluster", "Agg_Cluster"]
df.loc[sample_idx, cols]

## Step 10 — Save Output

In [ ]:
df.to_csv("dataset/segmented_wholesale_customers.csv", index=False)
print("Saved to dataset/segmented_wholesale_customers.csv")